# 07: Scaling and Performance Envelope

**What this measures when run:** Local write/query and assistant/RL-shaped workload timings for the selected mode, machine and dependencies. It does not prove production readiness.

**Inputs:** Stress tests with varying record counts, episode lengths, budgets

**Outputs:** Timing percentiles, operation/record counts, throughput tables and figures; no RAM profile or event-log bundle

**Runtime:** ~10 min (demo) / ~60 min (paper)

**Goal:** Measure local operation costs and scaling, with enough run context to interpret the results.


## 1. Setup

In [ ]:
RUN_MODE = "demo"  # @param ["demo", "paper"]

In [ ]:
import sys
from pathlib import Path
import time

NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name != "notebooks":
    NOTEBOOK_DIR = NOTEBOOK_DIR.parent
sys.path.insert(0, str(NOTEBOOK_DIR))

from _utils import (
    NotebookConfig, print_config, print_repro_stamp, set_all_seeds,
    RunManager, bootstrap_ci,
    setup_plotting, add_caption, COLORS, POLICY_COLORS,
)

from reckon.core.resources import MemoryBudget
from reckon.memory import CausalMemoryManager, PayloadType
from reckon.memory import FIFOPolicy, LRUPolicy, UtilityPolicy, EvidencePerBytePolicy

import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict

setup_plotting()
print("Imports OK!")

In [ ]:
config = NotebookConfig(
    notebook_name="07_scaling_perf",
    run_mode=RUN_MODE,
    seed=42,
    budget_slots=1000,
    task_name="benchmark",
    num_episodes=5 if RUN_MODE == "paper" else 2,
)
print_config(config)
print_repro_stamp(config)
set_all_seeds(config.seed)

## 2. Problem + Hypotheses

Production deployment requires understanding:

- **Latency**: How fast are write/read/prune operations?
- **Throughput**: How many records/second can we handle?
- **Memory**: How does RAM usage scale with records?
- **Policy overhead**: Do complex policies add significant cost?

**Workload Types:**

- **Synthetic micro-benchmarks**: Isolated operation testing
- **Assistant workload**: Conversation episodes with tool use
- **RL workload**: MiniGrid-style environment rollouts

**H1:** Write operations are O(1) amortized.

**H2:** How does write latency, including Python candidate preparation and Rust ranking, removal and insertion, scale under pruning pressure? This notebook does not isolate prune latency.

**H3:** EvidencePerBytePolicy has <10% overhead vs FIFO.

**H4:** Assistant workloads scale linearly with episode length.

These are questions for observed results, not guaranteed outcomes. Kernel policy ranking and Python query filtering are part of the timed path; a Rust data-structure bound is not the complexity of the whole CMM operation. RAM profiling remains a next test.


## 3. Minimal Theory

### Which path is timed?

| Operation | Current composition |
| --- | --- |
| write | Python serialization/record construction, overflow calculation and possible candidate selection, followed by Rust insertion/index accounting |
| query | Python cache scan/filter, utility sort, limit and access updates |
| prune-related write work | Python candidate preparation, Rust policy ranking and selection for joint byte/slot/token pressure, Python validation and Rust removals; no isolated prune benchmark here |
| score update | Python score update and event call; utility updates also refresh the native index |

The Rust store uses a HashMap and priority index with locks. Queries filter the Python cache, and writes serialize their payloads. Include those costs when interpreting the timings. See the [architecture](../ARCHITECTURE.md) and [memory manager design](../docs/08-Causal-Memory-Manager-Design.md).

### Workload shape

The assistant example performs three memory reads per step plus two or three writes, with an initial context record. The RL example writes observations/actions and optional rewards, with one memory read per step. These are synthetic operation patterns, not fixed read/write ratios or a trained policy. Retained memory does not choose the scripted/random actions.


## 4. System Mapping

```mermaid
flowchart LR
    subgraph BENCH["Benchmark Suite"]
        B1["Micro: Write/Read"]
        B2["Assistant Workload"]
        B3["RL Workload"]
    end

    subgraph CMM["CausalMemoryManager"]
        subgraph PYTHON["Python Layer"]
            P1["API facade"]
            P2["Event emission"]
        end

        subgraph RUST["Rust Core"]
            R1["RecordStore"]
            R2["EventLogger"]
        end

        PYTHON --> RUST
    end

    subgraph METRICS["Collected Metrics"]
        M1["Latency percentiles"]
        M2["Throughput (ops/sec)"]
        M3["Full-step latency (ms/step)"]
        M4["Operation and retained-record counts"]
    end

    BENCH --> CMM
    CMM --> METRICS
```

## 5. Experiment Design

1. **Micro-benchmarks:** Timed writes with pruning pressure and tag-query latency. There is no isolated prune timing or RAM measurement.
2. **Assistant-shaped workload:** Demo lengths: 8/16/32; paper adds: 64/128. Scripted tool-response patterns exercise memory operations, without a live LLM.
3. **RL-shaped workload:** Demo caps: 50/100 steps; paper adds: 200/500. An environment may terminate sooner; returned `total_steps` records what ran. Real environments and replay fallbacks remain separate.

**Micro scale points:** Demo: 100/500/1000/5000; paper additionally: 10000/50000/100000. These are configuration choices, not completed results.

**Collected metrics:** p50/p95/p99 operation or step times, throughput and operation/retained-record counts. Persistence cost, log-file size and RAM profiling are future measurements. Record the actual mode, environment source and exported arrays before drawing a scaling conclusion.

## 6. Demo Run: Micro-benchmarks

In [ ]:
def benchmark_write(n_records: int, budget_slots: int, policy) -> dict:
    """
    Benchmark write operation throughput.
    
    Returns:
        dict with latencies and throughput
    """
    budget = MemoryBudget(max_slots=budget_slots, max_bytes=budget_slots * 100)
    cmm = CausalMemoryManager(budget, policy, "bench-write")
    
    latencies = []
    
    for i in range(n_records):
        cmm.set_step(i)
        
        start = time.perf_counter()
        cmm.write(
            payload={"index": i, "data": "x" * 50},
            payload_type=PayloadType.OBSERVATION,
            tags=[f"tag_{i % 10}", "benchmark"],
            initial_utility=i / n_records,
        )
        end = time.perf_counter()
        
        latencies.append((end - start) * 1000)  # ms
    
    total_time = sum(latencies)
    
    return {
        "n_records": n_records,
        "total_ms": total_time,
        "throughput": n_records / (total_time / 1000),  # ops/sec
        "p50_ms": np.percentile(latencies, 50),
        "p95_ms": np.percentile(latencies, 95),
        "p99_ms": np.percentile(latencies, 99),
        "final_slots": cmm.current_usage_slots,
    }


# Quick demo
result = benchmark_write(100, budget_slots=50, policy=FIFOPolicy())
print(f"Write benchmark (100 records, budget=50):")
print(f"  Throughput: {result['throughput']:.0f} ops/sec")
print(f"  p50: {result['p50_ms']:.3f} ms")
print(f"  p95: {result['p95_ms']:.3f} ms")
print(f"  Final slots: {result['final_slots']}")

In [ ]:
def benchmark_read(n_records: int, n_queries: int, budget_slots: int) -> dict:
    """
    Benchmark read operation latency.
    """
    budget = MemoryBudget(max_slots=budget_slots, max_bytes=budget_slots * 100)
    cmm = CausalMemoryManager(budget, FIFOPolicy(), "bench-read")
    
    # Populate memory
    for i in range(n_records):
        cmm.set_step(i)
        cmm.write(
            payload={"index": i},
            payload_type=PayloadType.OBSERVATION,
            tags=[f"tag_{i % 10}", "benchmark"],
        )
    
    # Benchmark reads
    latencies = []
    
    for q in range(n_queries):
        tag = f"tag_{q % 10}"
        
        start = time.perf_counter()
        hits = cmm.retrieve(query={"tags": [tag]}, limit=5)
        end = time.perf_counter()
        
        latencies.append((end - start) * 1000)  # ms
    
    return {
        "n_records": n_records,
        "n_queries": n_queries,
        "p50_ms": np.percentile(latencies, 50),
        "p95_ms": np.percentile(latencies, 95),
        "p99_ms": np.percentile(latencies, 99),
        "avg_hits": len(hits),
    }


# Quick demo
result = benchmark_read(100, n_queries=50, budget_slots=100)
print(f"Read benchmark (100 records, 50 queries):")
print(f"  p50: {result['p50_ms']:.3f} ms")
print(f"  p95: {result['p95_ms']:.3f} ms")

## 7. Full Run: Multi-Workload Scaling

In [ ]:
# Scale points for different workloads
if RUN_MODE == "paper":
    MICRO_SCALE_POINTS = [100, 500, 1000, 5000, 10000, 50000, 100000]
    ASSISTANT_EPISODE_LENGTHS = [8, 16, 32, 64, 128]
    RL_EPISODE_LENGTHS = [50, 100, 200, 500]
else:
    MICRO_SCALE_POINTS = [100, 500, 1000, 5000]
    ASSISTANT_EPISODE_LENGTHS = [8, 16, 32]
    RL_EPISODE_LENGTHS = [50, 100]

POLICIES = {
    "FIFO": FIFOPolicy(),
    "LRU": LRUPolicy(),
    "Utility": UtilityPolicy(),
    "Evidence per byte": EvidencePerBytePolicy(0.5),
}

print(f"Running scaling benchmarks...")
print(f"Micro scale points: {MICRO_SCALE_POINTS}")
print(f"Assistant episode lengths: {ASSISTANT_EPISODE_LENGTHS}")
print(f"RL episode lengths: {RL_EPISODE_LENGTHS}")
print(f"Policies: {list(POLICIES.keys())}")

### 7.1 Micro-benchmark: Write Throughput by Policy

In [ ]:
# Write throughput by policy and scale
write_results = []

for policy_name, policy in POLICIES.items():
    for n in MICRO_SCALE_POINTS:
        # Budget at 50% of records to force pruning
        budget = n // 2
        
        result = benchmark_write(n, budget_slots=budget, policy=policy)
        result["policy"] = policy_name
        write_results.append(result)
        
        print(f"  {policy_name} @ {n}: {result['throughput']:.0f} ops/sec")

print("Done!")

### 7.2 Micro-benchmark: Read Latency Scaling

In [ ]:
# Read latency scaling
read_results = []

for n in MICRO_SCALE_POINTS:
    result = benchmark_read(n, n_queries=100, budget_slots=n)
    read_results.append(result)
    print(f"  Read @ {n}: p50={result['p50_ms']:.3f}ms, p95={result['p95_ms']:.3f}ms")

print("Done!")

### 7.3 Assistant Workload Scaling

In [ ]:
# Import tool environment for assistant workload
from _utils.adapters import (
    ToolEnv,
    ToolEnvConfig,
    get_demo_episodes,
)

def benchmark_assistant_workload(
    n_steps: int,
    budget_slots: int,
    policy,
    n_episodes: int = 3,
    seed: int = 42,
) -> dict:
    """
    Benchmark assistant (conversation) workload.

    Simulates a realistic conversation with:
    - Constraints to remember
    - Tool calls to execute
    - Multiple records per step

    Returns:
        dict with step latencies and throughput metrics
    """
    budget = MemoryBudget(max_slots=budget_slots, max_bytes=budget_slots * 200)
    cmm = CausalMemoryManager(budget, policy, f"assistant-{n_steps}")

    # Configure environment with longer episodes
    env_config = ToolEnvConfig(n_steps=n_steps)
    env = ToolEnv(config=env_config)

    step_latencies = []
    write_counts = []
    read_counts = []
    total_writes = 0
    total_reads = 0

    for ep in range(n_episodes):
        obs, info = env.reset(seed=seed + ep)

        # Write initial context
        cmm.set_step(0)
        cmm.write(
            payload={"type": "context", "episode": ep, "constraints": info.get("constraints", {})},
            payload_type=PayloadType.CUE,
            tags=["episode_start", "constraints"],
            initial_utility=1.0,
        )
        total_writes += 1

        step = 0
        terminated = False

        while not terminated and step < n_steps:
            step_start = time.perf_counter()
            cmm.set_step(step)

            step_writes = 0
            step_reads = 0

            # Three reads per step; writes per step vary below.
            for _ in range(3):
                cmm.retrieve(query={"tags": ["constraints"]}, limit=5)
                step_reads += 1

            # Write user message
            cmm.write(
                payload={"role": "user", "content": obs[:100] if obs else "", "step": step},
                payload_type=PayloadType.OBSERVATION,
                tags=["user_message", f"step_{step}"],
                initial_utility=0.8,
            )
            step_writes += 1

            # Simulate agent response with tool use
            agent_response = {
                "tool": {"name": "calculator", "args": {"expr": "42+1"}} if step % 3 == 0 else None,
                "final": f"Response for step {step}",
                "memory_write": [{"tag": "note", "content": f"Note {step}"}] if step % 2 == 0 else [],
            }

            # Write agent response
            cmm.write(
                payload={"role": "assistant", "response": agent_response, "step": step},
                payload_type=PayloadType.ACTION,
                tags=["assistant_response", f"step_{step}"],
                initial_utility=0.6,
            )
            step_writes += 1

            # Write memory notes if any
            for mem in agent_response.get("memory_write", []):
                cmm.write(
                    payload=mem,
                    payload_type=PayloadType.CUE,
                    tags=["agent_note", mem.get("tag", "note")],
                    initial_utility=0.9,
                )
                step_writes += 1

            # Step environment
            obs, reward, terminated, truncated, info = env.step(agent_response)

            step_end = time.perf_counter()
            step_latencies.append((step_end - step_start) * 1000)  # ms
            write_counts.append(step_writes)
            read_counts.append(step_reads)
            total_writes += step_writes
            total_reads += step_reads

            step += 1

    return {
        "workload": "assistant",
        "n_steps": n_steps,
        "n_episodes": n_episodes,
        "budget_slots": budget_slots,
        "total_steps": len(step_latencies),
        "total_writes": total_writes,
        "total_reads": total_reads,
        "step_p50_ms": np.percentile(step_latencies, 50),
        "step_p95_ms": np.percentile(step_latencies, 95),
        "step_p99_ms": np.percentile(step_latencies, 99),
        "avg_writes_per_step": np.mean(write_counts),
        "avg_reads_per_step": np.mean(read_counts),
        "throughput_steps_per_sec": len(step_latencies) / (sum(step_latencies) / 1000),
        "final_slots": cmm.current_usage_slots,
    }


# Run assistant workload benchmarks
assistant_results = []

print("Running assistant workload scaling...")
for policy_name, policy in POLICIES.items():
    for n_steps in ASSISTANT_EPISODE_LENGTHS:
        # Budget scales with episode length
        budget = max(50, n_steps * 4)

        result = benchmark_assistant_workload(
            n_steps=n_steps,
            budget_slots=budget,
            policy=policy,
            n_episodes=config.num_episodes,
            seed=config.seed,
        )
        result["policy"] = policy_name
        assistant_results.append(result)

        print(f"  {policy_name} @ {n_steps} steps: {result['step_p95_ms']:.2f}ms p95, {result['throughput_steps_per_sec']:.0f} steps/sec")

print("Done!")

### 7.4 RL Workload Scaling

In [ ]:
# Import RL environment adapters
from _utils.adapters import (
    get_env,
    check_env_available,
    obs_to_record,
    get_demo_traces,
    ReplayTraceEnv,
)


def benchmark_rl_workload(
    n_steps: int,
    budget_slots: int,
    policy,
    n_episodes: int = 3,
    seed: int = 42,
    env_name: str = "minigrid_memory",
) -> dict:
    """
    Benchmark RL workload with MiniGrid-style observations.
    
    Simulates an RL rollout with:
    - Observation encoding to memory
    - Action recording
    - Reward tracking
    
    Returns:
        dict with step latencies and throughput metrics
    """
    budget = MemoryBudget(max_slots=budget_slots, max_bytes=budget_slots * 300)
    cmm = CausalMemoryManager(budget, policy, f"rl-{n_steps}")
    
    # Get environment (real or fallback)
    try:
        env, is_real = get_env(env_name, fallback=True, verbose=False)
    except (ValueError, ImportError):
        # Use replay traces
        traces = get_demo_traces("memory")
        env = ReplayTraceEnv(traces, env_id="benchmark-replay")
        is_real = False
    
    step_latencies = []
    write_latencies = []
    read_latencies = []
    total_writes = 0
    total_reads = 0
    
    for ep in range(n_episodes):
        obs, info = env.reset(seed=seed + ep)
        
        # Write initial observation
        cmm.set_step(0)
        record_payload = obs_to_record(obs, info, step=0, env_id=env_name)
        
        write_start = time.perf_counter()
        cmm.write(
            payload=record_payload,
            payload_type=PayloadType.OBSERVATION,
            tags=["obs", "initial", f"ep_{ep}"],
            initial_utility=1.0,
        )
        write_latencies.append((time.perf_counter() - write_start) * 1000)
        total_writes += 1
        
        step = 0
        terminated = False
        truncated = False
        
        while not terminated and not truncated and step < n_steps:
            step_start = time.perf_counter()
            cmm.set_step(step)
            
            # Read memory for policy decision (1:1 ratio for RL)
            read_start = time.perf_counter()
            recent_obs = cmm.retrieve(query={"tags": ["obs"]}, limit=5)
            read_latencies.append((time.perf_counter() - read_start) * 1000)
            total_reads += 1
            
            # Select action (random or from trace)
            action = env.sample_action() if hasattr(env, "sample_action") else 0
            
            # Write action
            write_start = time.perf_counter()
            cmm.write(
                payload={"action": int(action) if hasattr(action, "__int__") else action, "step": step},
                payload_type=PayloadType.ACTION,
                tags=["action", f"step_{step}"],
                initial_utility=0.5,
            )
            write_latencies.append((time.perf_counter() - write_start) * 1000)
            total_writes += 1
            
            # Step environment
            obs, reward, terminated, truncated, info = env.step(action)
            
            # Write new observation
            record_payload = obs_to_record(obs, info, step=step + 1, env_id=env_name)
            
            write_start = time.perf_counter()
            cmm.write(
                payload=record_payload,
                payload_type=PayloadType.OBSERVATION,
                tags=["obs", f"step_{step + 1}"],
                initial_utility=0.8,
            )
            write_latencies.append((time.perf_counter() - write_start) * 1000)
            total_writes += 1
            
            # Write reward if non-zero
            if reward != 0:
                cmm.write(
                    payload={"reward": float(reward), "step": step},
                    payload_type=PayloadType.REWARD,
                    tags=["reward", f"step_{step}"],
                    initial_utility=0.9 if reward > 0 else 0.3,
                )
                total_writes += 1
            
            step_end = time.perf_counter()
            step_latencies.append((step_end - step_start) * 1000)
            
            step += 1
    
    env.close() if hasattr(env, "close") else None
    
    return {
        "workload": "rl",
        "env_name": env_name,
        "is_real_env": is_real if "is_real" in dir() else False,
        "n_steps": n_steps,
        "n_episodes": n_episodes,
        "budget_slots": budget_slots,
        "total_steps": len(step_latencies),
        "total_writes": total_writes,
        "total_reads": total_reads,
        "step_p50_ms": np.percentile(step_latencies, 50) if step_latencies else 0,
        "step_p95_ms": np.percentile(step_latencies, 95) if step_latencies else 0,
        "step_p99_ms": np.percentile(step_latencies, 99) if step_latencies else 0,
        "write_p50_ms": np.percentile(write_latencies, 50) if write_latencies else 0,
        "write_p95_ms": np.percentile(write_latencies, 95) if write_latencies else 0,
        "read_p50_ms": np.percentile(read_latencies, 50) if read_latencies else 0,
        "read_p95_ms": np.percentile(read_latencies, 95) if read_latencies else 0,
        "throughput_steps_per_sec": len(step_latencies) / (sum(step_latencies) / 1000) if step_latencies else 0,
        "final_slots": cmm.current_usage_slots,
    }


# Run RL workload benchmarks
rl_results = []

print("Running RL workload scaling...")
for policy_name, policy in POLICIES.items():
    for n_steps in RL_EPISODE_LENGTHS:
        # Budget scales with episode length (RL has more obs per step)
        budget = max(100, n_steps * 3)
        
        result = benchmark_rl_workload(
            n_steps=n_steps,
            budget_slots=budget,
            policy=policy,
            n_episodes=config.num_episodes,
            seed=config.seed,
        )
        result["policy"] = policy_name
        rl_results.append(result)
        
        print(f"  {policy_name} @ {n_steps} steps: {result['step_p95_ms']:.2f}ms p95, {result['throughput_steps_per_sec']:.0f} steps/sec")

print("Done!")

## 8. Results

### 8.1 Micro-benchmark Results

In [ ]:
# Plot: Write throughput by policy
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Throughput vs scale
ax = axes[0]
for i, policy_name in enumerate(POLICIES.keys()):
    policy_data = [r for r in write_results if r["policy"] == policy_name]
    ns = [r["n_records"] for r in policy_data]
    throughputs = [r["throughput"] for r in policy_data]

    ax.plot(ns, throughputs, 'o-', label=policy_name,
            color=list(COLORS.values())[i], linewidth=2, markersize=8)

ax.set_xlabel("Number of Records")
ax.set_ylabel("Throughput (ops/sec)")
ax.set_title("Write Throughput Scaling")
ax.legend()
ax.set_xscale("log")

# Right: p95 latency by policy
ax = axes[1]
for i, policy_name in enumerate(POLICIES.keys()):
    policy_data = [r for r in write_results if r["policy"] == policy_name]
    ns = [r["n_records"] for r in policy_data]
    p95s = [r["p95_ms"] for r in policy_data]

    ax.plot(ns, p95s, 'o-', label=policy_name,
            color=list(COLORS.values())[i], linewidth=2, markersize=8)

ax.set_xlabel("Number of Records")
ax.set_ylabel("p95 Latency (ms)")
ax.set_title("Write Latency Scaling (p95)")
ax.legend()
ax.set_xscale("log")

add_caption(axes[0], "Observed write throughput for this run and scale configuration", n=len(write_results))

plt.tight_layout()
plt.show()

In [ ]:
# Plot: Read latency scaling
fig, ax = plt.subplots(figsize=(10, 6))

ns = [r["n_records"] for r in read_results]
p50s = [r["p50_ms"] for r in read_results]
p95s = [r["p95_ms"] for r in read_results]
p99s = [r["p99_ms"] for r in read_results]

ax.plot(ns, p50s, 'o-', label="p50", color=COLORS["primary"], linewidth=2)
ax.plot(ns, p95s, 's-', label="p95", color=COLORS["secondary"], linewidth=2)
ax.plot(ns, p99s, '^-', label="p99", color=COLORS["warning"], linewidth=2)

ax.set_xlabel("Number of Records")
ax.set_ylabel("Read Latency (ms)")
ax.set_title("Read Latency Scaling")
ax.legend()
ax.set_xscale("log")

add_caption(ax, "Observed read-latency percentiles for this run", n=len(read_results))

plt.tight_layout()
plt.show()

### 8.2 Assistant Workload Results

In [ ]:
# Plot: Assistant workload step latency by episode length
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Step latency p95 vs episode length
ax = axes[0]
for i, policy_name in enumerate(POLICIES.keys()):
    policy_data = [r for r in assistant_results if r["policy"] == policy_name]
    steps = [r["n_steps"] for r in policy_data]
    p95s = [r["step_p95_ms"] for r in policy_data]

    ax.plot(steps, p95s, 'o-', label=policy_name,
            color=list(COLORS.values())[i], linewidth=2, markersize=8)

ax.set_xlabel("Episode Length (steps)")
ax.set_ylabel("Step Latency p95 (ms)")
ax.set_title("Assistant Workload: Step Latency")
ax.legend()

# Right: Throughput vs episode length
ax = axes[1]
for i, policy_name in enumerate(POLICIES.keys()):
    policy_data = [r for r in assistant_results if r["policy"] == policy_name]
    steps = [r["n_steps"] for r in policy_data]
    throughputs = [r["throughput_steps_per_sec"] for r in policy_data]

    ax.plot(steps, throughputs, 'o-', label=policy_name,
            color=list(COLORS.values())[i], linewidth=2, markersize=8)

ax.set_xlabel("Episode Length (steps)")
ax.set_ylabel("Throughput (steps/sec)")
ax.set_title("Assistant Workload: Throughput")
ax.legend()

add_caption(axes[0], "Observed assistant-shaped workload timings by configured length", n=len(assistant_results))

plt.tight_layout()
plt.show()

### 8.3 RL Workload Results

In [ ]:
# Plot: RL workload step latency by episode length
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Step latency p95 vs episode length
ax = axes[0]
for i, policy_name in enumerate(POLICIES.keys()):
    policy_data = [r for r in rl_results if r["policy"] == policy_name]
    steps = [r["n_steps"] for r in policy_data]
    p95s = [r["step_p95_ms"] for r in policy_data]

    ax.plot(steps, p95s, 'o-', label=policy_name,
            color=list(COLORS.values())[i], linewidth=2, markersize=8)

ax.set_xlabel("Episode Length (steps)")
ax.set_ylabel("Step Latency p95 (ms)")
ax.set_title("RL Workload: Step Latency")
ax.legend()

# Right: Write vs Read latency comparison
ax = axes[1]
evidence_data = [r for r in rl_results if r["policy"] == "Evidence per byte"]
steps = [r["n_steps"] for r in evidence_data]
write_p95 = [r["write_p95_ms"] for r in evidence_data]
read_p95 = [r["read_p95_ms"] for r in evidence_data]

ax.plot(steps, write_p95, 'o-', label="Write p95", color=COLORS["primary"], linewidth=2)
ax.plot(steps, read_p95, 's-', label="Read p95", color=COLORS["secondary"], linewidth=2)

ax.set_xlabel("Episode Length (steps)")
ax.set_ylabel("Latency p95 (ms)")
ax.set_title("RL Workload: Write vs Read Latency (Evidence per byte Policy)")
ax.legend()

add_caption(axes[0], "Observed RL-shaped workload timings; inspect actual step counts and environment mode", n=len(rl_results))

plt.tight_layout()
plt.show()

### 8.4 Policy Overhead Analysis

In [ ]:
# Policy overhead analysis
print("Policy Overhead Analysis (Micro-benchmarks):")
print("=" * 60)

for n in MICRO_SCALE_POINTS:
    fifo_result = next(r for r in write_results if r["policy"] == "FIFO" and r["n_records"] == n)
    fifo_throughput = fifo_result["throughput"]
    
    print(f"\n{n} records:")
    for policy_name in POLICIES.keys():
        policy_result = next(r for r in write_results if r["policy"] == policy_name and r["n_records"] == n)
        overhead = (1 - policy_result["throughput"] / fifo_throughput) * 100
        print(f"  {policy_name}: {policy_result['throughput']:.0f} ops/sec (overhead: {overhead:+.1f}%)")

In [ ]:
# Workload comparison summary
print("\nWorkload Comparison Summary:")
print("=" * 80)

# Assistant workload
print("\nAssistant Workload (Evidence per byte Policy):")
print(f"{'Episode Length':<15} {'Step p95 (ms)':<15} {'Throughput':<15} {'Writes/Step':<15}")
print("-" * 60)
for r in [r for r in assistant_results if r["policy"] == "Evidence per byte"]:
    print(f"{r['n_steps']:<15} {r['step_p95_ms']:<15.2f} {r['throughput_steps_per_sec']:<15.0f} {r['avg_writes_per_step']:<15.1f}")

# RL workload
print("\nRL Workload (Evidence per byte Policy):")
print(f"{'Episode Length':<15} {'Step p95 (ms)':<15} {'Throughput':<15} {'Total Writes':<15}")
print("-" * 60)
for r in [r for r in rl_results if r["policy"] == "Evidence per byte"]:
    print(f"{r['n_steps']:<15} {r['step_p95_ms']:<15.2f} {r['throughput_steps_per_sec']:<15.0f} {r['total_writes']:<15}")

In [ ]:
# Summary table
print("\nPerformance Summary:")
print("=" * 70)
print(f"{'Scale':<10} {'FIFO ops/s':<15} {'Evidence per byte ops/s':<15} {'Overhead':<10} {'Read p95'}")
print("-" * 70)

for n in MICRO_SCALE_POINTS:
    fifo = next(r for r in write_results if r["policy"] == "FIFO" and r["n_records"] == n)
    evidence = next(r for r in write_results if r["policy"] == "Evidence per byte" and r["n_records"] == n)
    read = next(r for r in read_results if r["n_records"] == n)
    
    overhead = (1 - evidence["throughput"] / fifo["throughput"]) * 100
    
    print(f"{n:<10} {fifo['throughput']:<15.0f} {evidence['throughput']:<15.0f} {overhead:<10.1f}% {read['p95_ms']:.3f}ms")

## 9. Debugger Walkthrough

This benchmark exports timing metrics and figures, not a high-record-count event-log bundle. Inspect its exported arrays and recorded actual step counts here. There is no generated log to import solely because metrics exist.

For a separately captured log, follow the [native debugger quick start](../src/reckon/viz/README.md#quick-start) and the [event-log producer boundaries](../docs/12-Reproducibility.md). Bounded in-memory operation events do not themselves provide complete query/payload snapshots, a RAM profile or automatic full-run persistence.

## 10. Limitations + Next Steps

**Limitations:**
- Single-threaded benchmarks
- Synthetic workloads (not real LLM latency)
- No persistence overhead measured
- Replay traces used when real envs unavailable

**Next tests:**
- [ ] Multi-threaded concurrent access
- [ ] Real LLM workload traces (with Ollama)
- [ ] Persistence (JSONL write) overhead
- [ ] Memory profiling with tracemalloc
- [ ] GPU-based embedding lookup

**Interpreting a run:** Read the actual exported scale points, throughput, latency percentiles, policy overhead and workload step counts. Demo stops at 5000 micro records; the paper configuration offers 100000 but does not establish that those measurements have completed. No fixed throughput, sub-millisecond, overhead or production-readiness conclusion is supplied by this source.

Assistant/RL steps include synthetic or environment work, not only CMM overhead. Replayed traces are fixed inputs, and no persistence or RAM cost is measured. A deployment decision needs its own representative workload and qualified evidence.

Within a benchmark invocation, the CMM is reused across episodes. Its timings therefore include retained state and any resulting pruning, rather than independent empty-memory episode resets.

## Export Artifacts

In [ ]:
run = RunManager(config)
run.save_config()

# Save all metrics
run.save_metrics({
    "write_results": write_results,
    "read_results": read_results,
    "assistant_results": assistant_results,
    "rl_results": rl_results,
})

# Save throughput figure
fig, ax = plt.subplots(figsize=(10, 6))
for i, policy_name in enumerate(POLICIES.keys()):
    policy_data = [r for r in write_results if r["policy"] == policy_name]
    ax.plot([r["n_records"] for r in policy_data],
            [r["throughput"] for r in policy_data],
            'o-', label=policy_name, color=list(COLORS.values())[i])
ax.set_xlabel("Records")
ax.set_ylabel("Throughput (ops/sec)")
ax.set_xscale("log")
ax.legend()
run.save_figure(fig, "throughput_scaling")
plt.close(fig)

# Save workload comparison figure
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
# Assistant
ax = axes[0]
evidence_assistant = [r for r in assistant_results if r["policy"] == "Evidence per byte"]
ax.plot([r["n_steps"] for r in evidence_assistant],
        [r["step_p95_ms"] for r in evidence_assistant], 'o-')
ax.set_xlabel("Episode Length")
ax.set_ylabel("Step p95 (ms)")
ax.set_title("Assistant Workload")
# RL
ax = axes[1]
evidence_rl = [r for r in rl_results if r["policy"] == "Evidence per byte"]
ax.plot([r["n_steps"] for r in evidence_rl],
        [r["step_p95_ms"] for r in evidence_rl], 'o-')
ax.set_xlabel("Episode Length")
ax.set_ylabel("Step p95 (ms)")
ax.set_title("RL Workload")
plt.tight_layout()
run.save_figure(fig, "workload_comparison")
plt.close(fig)

# Summary
max_scale = max(MICRO_SCALE_POINTS)
fifo_max = next(r for r in write_results if r["policy"] == "FIFO" and r["n_records"] == max_scale)
evidence_max = next(r for r in write_results if r["policy"] == "Evidence per byte" and r["n_records"] == max_scale)

max_assistant = max(ASSISTANT_EPISODE_LENGTHS)
assistant_evidence_max = next(r for r in assistant_results if r["policy"] == "Evidence per byte" and r["n_steps"] == max_assistant)

max_rl = max(RL_EPISODE_LENGTHS)
rl_evidence_max = next(r for r in rl_results if r["policy"] == "Evidence per byte" and r["n_steps"] == max_rl)

run.write_summary({
    "max_scale_tested": max_scale,
    "fifo_throughput_at_max": fifo_max["throughput"],
    "evidence_throughput_at_max": evidence_max["throughput"],
    "evidence_overhead_pct": (1 - evidence_max["throughput"] / fifo_max["throughput"]) * 100,
    "read_p95_at_max": read_results[-1]["p95_ms"],
    # Assistant workload
    "assistant_max_steps": max_assistant,
    "assistant_step_p95_ms": assistant_evidence_max["step_p95_ms"],
    "assistant_throughput": assistant_evidence_max["throughput_steps_per_sec"],
    # RL workload
    "rl_max_steps": max_rl,
    "rl_step_p95_ms": rl_evidence_max["step_p95_ms"],
    "rl_throughput": rl_evidence_max["throughput_steps_per_sec"],
})

run.print_artifacts()